# 48. Pydantic, FastAPI, and a tested inference endpoint

**Level:** Intermediate to advanced

**Prerequisites:** 05 and 41: HTTP and strict schemas.

[Course index](00_START_HERE.ipynb) | [Previous](47_Multimodal_Models_and_Contrastive_Learning.ipynb) | [Next](49_MLOps_Drift_and_Model_Artifacts.ipynb)

**Topics:** Pydantic, FastAPI, REST APIs, Inference endpoint, Request validation, Testing.

**Matching script:** `examples\48_Pydantic_FastAPI_and_Model_Serving.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

A serving layer accepts inputs, validates them, calls a model, and returns a documented output.
Training should normally not run for every request. The served preprocessing and model version
must match what was evaluated. JSON shape, numeric range, and domain validity are separate checks.

Pydantic v2 models validate data at a boundary. Strict mode avoids some implicit conversions;
extra='forbid' catches unexpected fields. FastAPI uses request/response models to define API
contracts and returns validation errors for malformed requests.

This lesson uses TestClient, an in-process client, and no listening server. It serves a clearly
labeled deterministic rule instead of pretending a threshold is a trained AI model. A trained
estimator could replace that callable after adding versioned preprocessing and evaluation.

Production needs authentication, authorization, TLS termination, input size limits, monitoring,
rate limiting, timeouts, concurrency/resource controls, and privacy-aware logging. A notebook
endpoint definition is not a hardened deployment.

## Mathematics or data contract

A service contract maps validated features to a response schema including prediction and model_version.

## Important limitation

The rule model is educational; TestClient verifies API behavior, not deployment resilience or model quality.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Test both a valid request and rejected input

**Purpose:** Exercise a real local API contract without opening a port.

### Read the code in this order
1. Define strict bounded request and response models.
2. Create a POST prediction route.
3. Call it with TestClient.
4. Assert the valid response and a 422 for an invalid feature.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.
- `with`/`async with` bounds a resource lifetime or a temporary execution mode.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
valid: {'label': 1, 'model_version': 'teaching-rule-v1'} invalid status: 422
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, ConfigDict, Field

class Features(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    latency_ms: float = Field(ge=0, le=10000, allow_inf_nan=False)

class Prediction(BaseModel):
    label: int
    model_version: str

app = FastAPI()

@app.post("/predict", response_model=Prediction)
def predict(features: Features):
    return Prediction(label=int(features.latency_ms >= 20), model_version="teaching-rule-v1")

with TestClient(app) as client:
    good = client.post("/predict", json={"latency_ms": 25.0})
    bad = client.post("/predict", json={"latency_ms": -1.0})
    assert good.status_code == 200
    assert good.json() == {"label": 1, "model_version": "teaching-rule-v1"}
    assert bad.status_code == 422
print("valid:", good.json(), "invalid status:", bad.status_code)

valid: {'label': 1, 'model_version': 'teaching-rule-v1'} invalid status: 422


### Interpret example 1

The result should demonstrate this purpose: **Exercise a real local API contract without opening a port.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Swap in a previously fitted estimator and ensure preprocessing is loaded once, not fitted on each request.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Pydantic` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: A service contract maps validated features to a response schema including prediction and model_version.
- Name the boundary of the demonstration: The rule model is educational; TestClient verifies API behavior, not deployment resilience or model quality.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://fastapi.tiangolo.com/tutorial/testing/`
- `https://docs.pydantic.dev/latest/concepts/models/`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.